<a href="https://colab.research.google.com/github/vinhnq225/Project/blob/main/Hadoop_Ecommerce_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!java -version

[0.026s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.026s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [2]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  openjdk-17-jre-headless
Suggested packages:
  openjdk-17-demo openjdk-17-source libnss-mdns fonts-dejavu-extra
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  openjdk-17-jdk-headless openjdk-17-jre-headless
0 upgraded, 2 newly installed, 0 to remove and 32 not upgraded.
Need to get 120 MB of archives.
After this operation, 272 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 openjdk-17-jre-headless amd64 17.0.20.1+1-1~24.04 [48.1 MB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 open

In [12]:
!wget -q --show-progress \https://archive.apache.org/dist/hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz

hadoop-3.5.0.tar.gz 100%[===================>] 554.35M  1.37MB/s    in 2m 3s   


In [4]:
!tar -xzf hadoop-3.5.0.tar.gz


gzip: stdin: unexpected end of file
tar: Unexpected EOF in archive
tar: Unexpected EOF in archive
tar: Error is not recoverable: exiting now


In [5]:
import os
import subprocess

os.environ["HADOOP_HOME"] = "/content/hadoop-3.5.0"
os.environ["JAVA_HOME"] = os.path.realpath(
    subprocess.check_output(
        ["bash", "-lc", "dirname $(dirname $(readlink -f $(which java)))"],
        text=True
    ).strip()
)
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "/bin:" + os.environ["PATH"]

with open(os.environ["HADOOP_HOME"] + "/etc/hadoop/hadoop-env.sh", "a") as f:
    f.write("\nexport JAVA_HOME=" + os.environ["JAVA_HOME"] + "\n")

!hadoop version


ERROR: Invalid HADOOP_HDFS_HOME


In [17]:
!mkdir -p /content/hadoop_data/namenode
!mkdir -p /content/hadoop_data/datanode
!ls -ld /content/hadoop_data/*

drwxr-xr-x 2 root root 4096 Oct 14 06:03 /content/hadoop_data/datanode
drwxr-xr-x 2 root root 4096 Oct 14 06:03 /content/hadoop_data/namenode


In [7]:
!head orders_2026_09.csv

transaction_id,order_date,order_time,province,category,channel,quantity,unit_price,amount,status,shipping_days
TX001,2026-09-01,08:30,Ha Noi,Electronics,Web,1,3400000,3400000,SUCCESS,2
TX002,2026-09-01,09:00,Da Nang,Fashion,App,2,260000,520000,SUCCESS,3
TX003,2026-09-02,10:30,Can Tho,Grocery,Web,1,180000,180000,FAILED,4
TX004,2026-09-02,11:00,Lam Dong,Book,App,1,250000,250000,SUCCESS,2
TX005,2026-09-03,13:30,Ha Noi,Electronics,Web,1,2500000,2500000,SUCCESS,1


In [19]:
%%writefile /content/hadoop-3.5.0/etc/hadoop/core-site.xml
<?xml version="1.0"?>
<configuration>
    <property>
        <name>fs.defaultFS</name>
        <value>hdfs://localhost:9000</value>
    </property>
</configuration>

Overwriting /content/hadoop-3.5.0/etc/hadoop/core-site.xml


In [20]:
%%writefile /content/hadoop-3.5.0/etc/hadoop/hdfs-site.xml
<?xml version="1.0"?>
<configuration>
    <property>
        <name>dfs.replication</name>
        <value>1</value>
    </property>
    <property>
        <name>dfs.namenode.name.dir</name>
        <value>file:///content/hadoop_data/namenode</value>
    </property>
    <property>
        <name>dfs.datanode.data.dir</name>
        <value>file:///content/hadoop_data/datanode</value>
    </property>
</configuration>

Overwriting /content/hadoop-3.5.0/etc/hadoop/hdfs-site.xml


In [21]:
%%writefile /content/hadoop-3.5.0/etc/hadoop/mapred-site.xml
<?xml version="1.0"?>
<configuration>
    <property>
        <name>mapreduce.framework.name</name>
        <value>local</value>
    </property>
</configuration>

Overwriting /content/hadoop-3.5.0/etc/hadoop/mapred-site.xml


In [22]:
!hadoop getconf -confKey fs.defaultFS
!hadoop getconf -confKey dfs.replication
!hadoop getconf -confKey mapreduce.framework.name

ERROR: Invalid HADOOP_HDFS_HOME
ERROR: Invalid HADOOP_HDFS_HOME
ERROR: Invalid HADOOP_HDFS_HOME


In [23]:
!hdfs namenode -format -force

ERROR: Invalid HADOOP_HDFS_HOME


In [27]:
!hdfs --daemon start namenode
!hdfs --daemon start datanode
!jps


ERROR: Invalid HADOOP_HDFS_HOME
ERROR: Invalid HADOOP_HDFS_HOME
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
5198 Jps


In [25]:
!hdfs dfsadmin -report
!hdfs dfs -ls /

ERROR: Invalid HADOOP_HDFS_HOME
ERROR: Invalid HADOOP_HDFS_HOME


In [26]:
!hdfs dfs -mkdir -p /user/student/input
!hdfs dfs -mkdir -p /user/student/output
!hdfs dfs -ls -R /user/student

ERROR: Invalid HADOOP_HDFS_HOME
ERROR: Invalid HADOOP_HDFS_HOME
ERROR: Invalid HADOOP_HDFS_HOME


In [28]:
%%writefile /content/sales.txt
Hanoi,1200
Danang,850
HCM,1700
Hanoi,950

Writing /content/sales.txt


In [29]:
!hdfs dfs -put -f /content/sales.txt /user/student/input/
!hdfs dfs -ls -h /user/student/input

ERROR: Invalid HADOOP_HDFS_HOME
ERROR: Invalid HADOOP_HDFS_HOME


In [30]:
!hdfs dfs -cat /user/student/input/sales.txt
!hdfs dfs -head /user/student/input/sales.txt
!hdfs dfs -tail /user/student/input/sales.txt

ERROR: Invalid HADOOP_HDFS_HOME
ERROR: Invalid HADOOP_HDFS_HOME
ERROR: Invalid HADOOP_HDFS_HOME


In [31]:
!hdfs dfs -cp /user/student/input/sales.txt /user/student/input/sales_backup.txt
!hdfs dfs -mv /user/student/input/sales_backup.txt /user/student/input/sales_2026.txt
!hdfs dfs -ls /user/student/input

ERROR: Invalid HADOOP_HDFS_HOME
ERROR: Invalid HADOOP_HDFS_HOME
ERROR: Invalid HADOOP_HDFS_HOME


In [32]:
!hdfs dfs -du -h /user/student/input
!hdfs dfs -df -h /
!hdfs dfsadmin -report

ERROR: Invalid HADOOP_HDFS_HOME
ERROR: Invalid HADOOP_HDFS_HOME
ERROR: Invalid HADOOP_HDFS_HOME


In [33]:
!rm -f /content/sales_downloaded.txt
!hdfs dfs -get /user/student/input/sales.txt /content/sales_downloaded.txt
!cat /content/sales_downloaded.txt

ERROR: Invalid HADOOP_HDFS_HOME
cat: /content/sales_downloaded.txt: No such file or directory
